# ray-parametric-form — ex1: evaluate one ray at many parameter values

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `ray-parametric-form`. Running the final beacon cell reports progress against the `Geometry: Ray parametric form` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Geometry: Ray parametric form` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`ray-parametric-form`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "ray-parametric-form"
DD_SUBTOPIC = "Geometry: Ray parametric form"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Ray parametric form — quick refresher

A ray in 3-D is specified by an **origin** `O` (a point) and a **direction** `D` (a vector). Every point on the ray is

```
R(u) = O + u * D,  u >= 0
```

**Anatomy:**
- `u = 0` returns the origin.
- `u > 0` walks forward along `D`.
- `u < 0` would walk *backward* — by convention, real rays restrict `u >= 0`.

**Storage convention used in ARENA.** A ray is a `(2, 3)` tensor: row 0 is `O`, row 1 is `D`. A batch of `B` rays is a `(B, 2, 3)` tensor. To evaluate `B` rays at a single scalar `u`, you compute `origin + u * direction` with ordinary broadcasting; to evaluate one ray at `M` parameter values, you broadcast `(M,)` against `(3,)`.

**Why `D` is not required to be unit-length.** If `||D|| != 1` then `u` is not a metric distance — it's a parameter along `D`. ARENA leaves `D` un-normalized everywhere, so `u` is dimensionless.

### Exercise 1 — evaluate one ray at many parameter values

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the parametric ray equation `R(u) = O + u * D` to evaluate a single ray at a 1-D batch of parameter values, returning a tensor of points along the ray.
> Keywords: ray, parametric, broadcasting, visualization
> ```

**KCs targeted:** `ray-eval-broadcast-u`, `ray-origin-direction-storage`

Implement `ex1_eval_ray(ray, us)`.

- `ray` has shape `(2, 3)` — row 0 is the origin `O`, row 1 is the direction `D`.
- `us` has shape `(M,)` — the parameter values to evaluate at.
- Return shape `(M, 3)` where row `m` is `O + us[m] * D`.

**Hint.** Unpack the ray with one of the patterns you've seen (`ray[0], ray[1]` or `ray.unbind(dim=0)`). Then broadcast `us` against `D`: `us` is `(M,)`, `D` is `(3,)`, so reshape `us` to `(M, 1)` before multiplying so it lines up against the trailing 3.

The visualization plots the projected (x, z) trajectory of the ray points so you can verify the line goes through the origin and extends along `D`.

In [ ]:
def ex1_eval_ray(ray: Tensor, us: Tensor) -> Tensor:
    O, D = ray[0], ray[1]              # each (3,)
    return O + us.unsqueeze(-1) * D    # (M,1) * (3,) -> (M,3)


<details><summary>Solution</summary>

```python
def ex1_eval_ray(ray: Tensor, us: Tensor) -> Tensor:
    O, D = ray[0], ray[1]              # each (3,)
    return O + us.unsqueeze(-1) * D    # (M,1) * (3,) -> (M,3)
```

**Why `unsqueeze(-1)`.** Bare `us * D` would try to broadcast `(M,)` against `(3,)`, which only works when `M == 3` — a silent bug. Adding a trailing size-1 axis (`(M, 1)`) lines up explicitly against the trailing 3 of `D`.

**Why `O + ...` works without unsqueezing `O`.** Broadcasting lines up trailing dims: `(M, 3) + (3,) → (M, 3)`. `O` is automatically replicated across the leading `M` axis.

**Equivalent forms.** `O + t.outer(us, D)` also works and is arguably clearer about the rank-1 outer-product structure. Both are O(M*3) flops; pick whichever reads better in context.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()